In [ ]:
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

from core import (
    RNNConfig,
    MotorCortexRNN,
    BCIDecoderConfig,
    PopulationBCIDecoder,
    TrialInputConfig,
    CursorTargetConfig,
    TrainerConfig,
    BCITrainer,
)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# let set some global parameters

global_noise_std = 0.075
num_trials = 100
n_eval_every = 2
n_save_train_snapshot_every = 1

# set model configs

trial_cfg = TrialInputConfig(
    n_inp=10,
    t_baseline=20,
    t_task=20,
    t_late=20,
    baseline_scale=0.3,
    task_scale=1.0,
    late_scale=0.6,
    input_noise_std=global_noise_std,
    device="cpu",
)

target_cfg = CursorTargetConfig(
    baseline_target=0.0,
    late_target=1.0,
    task_target_mode="linear",
    baseline_weight=0.5,
    task_weight=1.0,
    late_weight=0.75,
    device="cpu",
)

In [ ]:
train_mode = 'input'      
trainer_cfg = TrainerConfig(
    train_mode= train_mode,      # all, input, recurrent
    lr=1e-3,
    grad_clip_norm=1.0,
    device="cpu",
    freeze_trial_inputs=True,
    input_scaling_mode="cursor_mean",
    input_scale_gain=0.2,
)
# Build model
rnn = MotorCortexRNN(RNNConfig(n_inp=10, n_rec=200, noise_std=global_noise_std, device="cpu"))
decoder = PopulationBCIDecoder(BCIDecoderConfig(n_rec=200, n_bci=50, device="cpu"))
trainer_input_model = BCITrainer(rnn, decoder, trial_cfg, target_cfg, trainer_cfg)
# Save first decoder axis
axis1 = trainer_input_model.decoder.axis.detach().clone()
# Phase 1: learn decoder 1
trainer_input_model.fit_phase(
    n_steps=num_trials,
    phase_name="decoder_1",
    start_step=0,
    batch_size=32,
    eval_every=n_eval_every,
    eval_batch_size=64,
    print_every=30,
    save_train_snapshot_every=n_save_train_snapshot_every,
    save_eval_snapshots=True,
)
# Make a second decoder axis on the SAME 50 neurons,
# and force it to be orthogonal to axis1
axis2 = torch.zeros_like(axis1)
idx = trainer_input_model.decoder.neuron_indices
v = torch.randn(len(idx), device=axis1.device)
axis2[idx] = v
# subtract projection onto axis1
axis2 = axis2 - torch.dot(axis2, axis1) * axis1
# normalize
axis2 = axis2 / axis2.norm()
# Remap: same model, same learned weights, new decoder
trainer_input_model.set_decoder_axis(axis2)
# Phase 2: relearn new decoder
trainer_input_model.fit_phase(
    n_steps=num_trials,
    phase_name="decoder_2_remap",
    start_step=num_trials,
    batch_size=32,
    eval_every=n_eval_every,
    eval_batch_size=64,
    print_every=10,
    save_train_snapshot_every=n_save_train_snapshot_every,   # save every remap trial if you want signatures
    save_eval_snapshots=True,
)

In [ ]:
train_mode = 'recurrent'      # all, input, recurrent
trainer_cfg = TrainerConfig(
    train_mode= train_mode,      # all, input, recurrent
    lr=1e-3,
    grad_clip_norm=1.0,
    device="cpu",
    freeze_trial_inputs=True,
    input_scaling_mode="cursor_mean",
    input_scale_gain=0.2,
)

# Build model
rnn = MotorCortexRNN(RNNConfig(n_inp=10, n_rec=200, noise_std=global_noise_std, device="cpu"))
decoder = PopulationBCIDecoder(BCIDecoderConfig(n_rec=200, n_bci=50, device="cpu"))
trainer_recurrent_model = BCITrainer(rnn, decoder, trial_cfg, target_cfg, trainer_cfg)
# Save first decoder axis
axis1 = trainer_recurrent_model.decoder.axis.detach().clone()
# Phase 1: learn decoder 1
trainer_recurrent_model.fit_phase(
    n_steps=num_trials,
    phase_name="decoder_1",
    start_step=0,
    batch_size=32,
    eval_every=n_eval_every,
    eval_batch_size=64,
    print_every=30,
    save_train_snapshot_every=n_save_train_snapshot_every,
    save_eval_snapshots=True,
)
# Make a second decoder axis on the SAME 50 neurons,
# and force it to be orthogonal to axis1
axis2 = torch.zeros_like(axis1)
idx = trainer_recurrent_model.decoder.neuron_indices
v = torch.randn(len(idx), device=axis1.device)
axis2[idx] = v
# subtract projection onto axis1
axis2 = axis2 - torch.dot(axis2, axis1) * axis1
# normalize
axis2 = axis2 / axis2.norm()
# Remap: same model, same learned weights, new decoder
trainer_recurrent_model.set_decoder_axis(axis2)
# Phase 2: relearn new decoder
trainer_recurrent_model.fit_phase(
    n_steps=num_trials,
    phase_name="decoder_2_remap",
    start_step=num_trials,
    batch_size=32,
    eval_every=n_eval_every,
    eval_batch_size=64,
    print_every=10,
    save_train_snapshot_every=n_save_train_snapshot_every,   # save every remap trial if you want signatures
    save_eval_snapshots=True,
)

In [ ]:
# let plot the results 
from core import (
    plot_trainer_summary,
    plot_geometry_metrics,
    plot_global_pca_phase_trajectories,
)

plot_trainer_summary(
    trainer=trainer_input_model,
    model_name="Input_Plasticity",
    axis_shift_step=num_trials,
    fig_size=(6, 2.5),
    save_path="results/input_vs_recurrent/trainer_summary",
)
plot_trainer_summary(
    trainer=trainer_recurrent_model,
    model_name="Recurrent_Plasticity",
    axis_shift_step=num_trials,
    fig_size=(6, 2.5),
    save_path="results/input_vs_recurrent/trainer_summary",
)

plot_geometry_metrics(
    trainers=[trainer_input_model, trainer_recurrent_model],
    trainer_names=["Input_Plasticity", "Recurrent_Plasticity"],
    axis_shift_step=num_trials,
    save_path="results/input_vs_recurrent/geometry_metrics"
)

plot_global_pca_phase_trajectories(
    trainer=trainer_input_model,
    axis_shift_step=num_trials,
    model_name="Input_Plasticity",
    show_colormap_strips=True,
    save_path="results/input_vs_recurrent",
)
plot_global_pca_phase_trajectories(
    trainer=trainer_recurrent_model,    
    axis_shift_step=num_trials,
    model_name="Recurrent_Plasticity",
    show_colormap_strips=True,
    save_path="results/input_vs_recurrent",
)